# Notebook 00 — Environment & Setup Check

**Goal:** verify your environment is ready to run the Phase-2 measurement pipeline before you waste time loading 20 models on a misconfigured host.

**Checks:**
1. Working directory is correct (you're in `notebooks/` under `code_active/`).
2. `vlm_audit` package is importable.
3. PyTorch is installed, sees CUDA, and the GPU has enough VRAM for the largest model in the roster.
4. HuggingFace `transformers` ≥ 4.46 (Idefics2 + LLaVA-NeXT minimum).
5. `calflops` / `fvcore` / `ptflops` (FLOPs counters) all import.
6. `bitsandbytes` available (only required if you plan to quantize).
7. `HF_TOKEN` is set in env (required for gated models: LLaMA / IDEFICS / Pixtral / Llama-3.2-Vision).
8. Free disk space ≥ 500 GB recommended (250-300 GB for HF cache; 2× doubles to ~500 GB).
9. Output directories created.
10. **Mini smoke test** — load the smallest model (ViLT-B/32, 117M) and run the param extractor end-to-end.

**Inputs:** none.
**Outputs:** none persisted; prints diagnostics inline.
**Compute:** CPU OK; GPU only for the mini smoke test (and that's optional).
**Runtime:** ~30 seconds with warm HF cache, ~3 minutes cold (downloads ViLT 460 MB).

Run this notebook FIRST before any of the measurement notebooks. If any check fails, fix it before proceeding.

---
## 1. Setup (Runpod)

In [ ]:
# %%capture
# !pip install -q -r ../requirements.txt

In [1]:
import os, sys, shutil
from pathlib import Path

# Set HF_TOKEN if not already in env. Gated models will fail without it.
# os.environ['HF_TOKEN'] = 'hf_...'  # uncomment + paste your token if needed

NB_DIR = Path.cwd()
CODE_ACTIVE = NB_DIR.parent if NB_DIR.name == 'notebooks' else NB_DIR
sys.path.insert(0, str(CODE_ACTIVE))
print(f'Working dir:        {NB_DIR}')
print(f'code_active dir:    {CODE_ACTIVE}')
print(f'sys.path[0] added:  {sys.path[0]}')

Working dir:        /workspace/code_active/notebooks
code_active dir:    /workspace/code_active
sys.path[0] added:  /workspace/code_active


---
## 2. Working directory + repo layout

We must be running inside `Thread_B_Architectural_Asymmetry/code_active/notebooks/` so that the `../` paths in subsequent notebooks resolve to the right `vlm_audit/`, `requirements.txt`, and `outputs/` directories.

In [2]:
# Verify required files exist relative to CODE_ACTIVE
REQUIRED_FILES = [
    CODE_ACTIVE / 'vlm_audit' / '__init__.py',
    CODE_ACTIVE / 'vlm_audit' / 'models.py',
    CODE_ACTIVE / 'vlm_audit' / 'components.py',
    CODE_ACTIVE / 'vlm_audit' / 'metrics.py',
    CODE_ACTIVE / 'vlm_audit' / 'runner.py',
    CODE_ACTIVE / 'requirements.txt',
]
missing = [p for p in REQUIRED_FILES if not p.exists()]
if missing:
    print('⛔ MISSING FILES — your CODE_ACTIVE path is wrong or repo is incomplete:')
    for p in missing:
        print(f'  - {p}')
    raise FileNotFoundError(f'Missing {len(missing)} required files')
else:
    print('✓ All required vlm_audit package files present')
    for p in REQUIRED_FILES:
        sz = p.stat().st_size
        print(f'  ✓ {p.name:25s} ({sz:>7,} bytes)')

✓ All required vlm_audit package files present
  ✓ __init__.py               (  1,626 bytes)
  ✓ models.py                 ( 11,587 bytes)
  ✓ components.py             ( 20,930 bytes)
  ✓ metrics.py                ( 19,280 bytes)
  ✓ runner.py                 ( 27,520 bytes)
  ✓ requirements.txt          (  1,687 bytes)


---
## 3. Python + PyTorch + CUDA

PyTorch must be ≥ 2.0. CUDA must be available for the memory analysis notebook (04). For everything else (params, FLOPs, tokens), CPU is sufficient but slower.

In [3]:
import torch
print(f'Python:            {sys.version.split()[0]}')
print(f'PyTorch:           {torch.__version__}')
print(f'CUDA available:    {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA version:      {torch.version.cuda}')
    n_gpus = torch.cuda.device_count()
    print(f'GPU count:         {n_gpus}')
    for i in range(n_gpus):
        props = torch.cuda.get_device_properties(i)
        vram = props.total_memory / 1e9
        print(f'  GPU {i}: {props.name}  ({vram:.1f} GB)')
        if vram < 24:
            print(f'    ⚠ GPU {i} VRAM < 24 GB — Pixtral-12B and DeepSeek-VL2 will OOM without quantization')
        elif vram < 48:
            print(f'    ⚠ GPU {i} VRAM 24-48 GB — fine for fp16 single-model loads, may struggle on DeepSeek-VL2 (30 GB)')
        else:
            print(f'    ✓ GPU {i} VRAM ≥ 48 GB — every model in the n=20 roster fits fp16')
else:
    print('⚠ CUDA unavailable — notebook 04 (memory analysis) will be skipped.')
    print('  All other notebooks can run CPU-only.')

Python:            3.12.3
PyTorch:           2.8.0+cu128
CUDA available:    True
CUDA version:      12.8
GPU count:         1
  GPU 0: NVIDIA A100-SXM4-80GB  (85.1 GB)
    ✓ GPU 0 VRAM ≥ 48 GB — every model in the n=20 roster fits fp16


---
## 4. HuggingFace transformers + dependencies

Minimum `transformers` version is 4.46 (for Idefics2, LLaVA-NeXT, Qwen2-VL). For Phase-2 models (Qwen2.5-VL, Chameleon, Janus-Pro, DeepSeek-VL2, InternVL3), some require `trust_remote_code=True` and the latest transformers commits — see the per-family handling in `vlm_audit/runner.py`.

In [4]:
import transformers
from packaging.version import parse as vparse
v = transformers.__version__
print(f'transformers: {v}')
min_v = '4.46.0'
max_v_excl = '5.0.0'  # transformers 5.x breaks custom-code models — see below
if vparse(v) < vparse(min_v):
    print(f'⛔ transformers {v} < {min_v} — upgrade required. Run: pip install "transformers>=4.46,<5.0"')
elif vparse(v) >= vparse(max_v_excl):
    print(f'⛔ transformers {v} >= 5.0 — too new!')
    print('   transformers 5.x renamed `_tied_weights_keys` → `all_tied_weights_keys`,')
    print('   which breaks every model using trust_remote_code modeling files:')
    print('     - Molmo-7B-D       AttributeError: no attribute all_tied_weights_keys')
    print('     - InternVL3-8B     same')
    print('     - DeepSeek-VL2     KeyError model_type=deepseek_vl_v2 (auto_map fallback removed)')
    print('     - Janus-Pro-7B     KeyError model_type=multi_modality (same)')
    print('   Plus calflops fails with `bool object has no attribute to` on every vision tower.')
    print('   Downgrade:  pip install "transformers>=4.46,<5.0"  (or pin to 4.49)')
else:
    print(f'✓ transformers {v} in supported range [{min_v}, {max_v_excl})')

# Required companion libs
for name in ['accelerate', 'safetensors', 'tokenizers']:
    try:
        mod = __import__(name)
        print(f'✓ {name}: {mod.__version__}')
    except ImportError:
        print(f'⛔ {name} NOT installed — run pip install {name}')

# timm — required for InternVL3's custom vision encoder
try:
    import timm
    print(f'✓ timm: {timm.__version__}  (needed for InternVL3 custom encoder)')
except ImportError:
    print('⛔ timm NOT installed — InternVL3 will fail with ImportError on load')
    print('   Run: pip install timm')

transformers: 4.57.6
✓ transformers 4.57.6 in supported range [4.46.0, 5.0.0)
✓ accelerate: 1.13.0
✓ safetensors: 0.7.0
✓ tokenizers: 0.22.2
✓ timm: 1.0.27  (needed for InternVL3 custom encoder)


In [5]:
# FLOPs counters — at least one must work; the runner tries them in order
flops_chain = []
for name in ['calflops', 'fvcore', 'ptflops']:
    try:
        __import__(name)
        flops_chain.append(name)
        print(f'✓ {name} importable')
    except ImportError as e:
        print(f'⚠ {name} not installed: {e}')
if not flops_chain:
    print('⛔ NO FLOPs counter installed — notebook 03 will fall back to manual estimate for every model')
else:
    print(f'FLOPs source priority chain: {flops_chain}  (calflops is preferred; verified on CLIP-L)')

# Quantization — optional (only needed if you plan to use --quantization)
try:
    import bitsandbytes
    print(f'✓ bitsandbytes {bitsandbytes.__version__}  (quantization available)')
except ImportError:
    print('⚠ bitsandbytes not installed — quantization unavailable. Run pip install bitsandbytes if you need 4bit/8bit loads.')

✓ calflops importable
✓ fvcore importable
✓ ptflops importable
FLOPs source priority chain: ['calflops', 'fvcore', 'ptflops']  (calflops is preferred; verified on CLIP-L)
✓ bitsandbytes 0.49.2  (quantization available)


---
## 5. HuggingFace token (gated models)

Six models in the roster require `HF_TOKEN` and have gated access:
* `meta-llama/Llama-3.2-11B-Vision-Instruct` (Llama 3.2 license)
* `mistral-community/pixtral-12b` (Mistral terms; using HF-converted port — original `mistralai/Pixtral-12B-2409` ships consolidated.safetensors and fails on transformers.from_pretrained)
* `HuggingFaceM4/idefics-9b` and `HuggingFaceM4/idefics2-8b` (research-only)
* `Salesforce/instructblip-vicuna-7b` (LLaMA derivative, gated)
* `facebook/chameleon-7b` (Meta research license)

If you haven't accepted each model's license on its HuggingFace page, the download will return a 403 Forbidden (`GatedRepoError`). Visit each model's HF page (e.g. https://huggingface.co/facebook/chameleon-7b) and click 'Agree and access repository' before re-running notebook 02. The next cell will list any gated repos that still 403.

In [6]:
hf_token = os.environ.get('HF_TOKEN', '')
if not hf_token:
    print('⛔ HF_TOKEN not set in env')
    print('   On Runpod: add HF_TOKEN to your pod env vars OR uncomment the os.environ line in §1')
    print('   Without it, 6 models (Llama-3.2-Vision, Pixtral, IDEFICS, Idefics2, InstructBLIP, Chameleon) will 403 on download.')
else:
    masked = hf_token[:4] + '...' + hf_token[-4:] if len(hf_token) > 12 else '***'
    print(f'✓ HF_TOKEN set in env  ({masked})')
    # Quick API sanity ping
    try:
        from huggingface_hub import HfApi
        api = HfApi()
        user = api.whoami(token=hf_token)
        print(f'  Authenticated as: {user.get("name", "?")}  (org: {user.get("orgs", [{}])[0].get("name", "none")})')
    except Exception as e:
        print(f'  ⚠ Could not call whoami: {e}')

# Probe each gated repo with a HEAD request to surface 403s upfront,
# rather than discovering them mid-loop in notebook 02.
GATED_REPOS = [
    'meta-llama/Llama-3.2-11B-Vision-Instruct',
    'mistral-community/pixtral-12b',
    'HuggingFaceM4/idefics-9b',
    'HuggingFaceM4/idefics2-8b',
    'Salesforce/instructblip-vicuna-7b',
    'facebook/chameleon-7b',
]
if hf_token:
    print()
    print('Probing gated repos for license acceptance:')
    from huggingface_hub import HfApi
    api = HfApi()
    fails = []
    for repo in GATED_REPOS:
        try:
            api.repo_info(repo, token=hf_token)
            print(f'  ✓ {repo}')
        except Exception as e:
            msg = str(e).splitlines()[0][:120]
            print(f'  ⛔ {repo}  →  {type(e).__name__}: {msg}')
            fails.append(repo)
    if fails:
        print()
        print(f'   {len(fails)} gated repo(s) inaccessible. Visit each URL and click Agree:')
        for repo in fails:
            print(f'     https://huggingface.co/{repo}')
        print('   Notebook 02 will skip these with a clean failure message.')

⛔ HF_TOKEN not set in env
   On Runpod: add HF_TOKEN to your pod env vars OR uncomment the os.environ line in §1
   Without it, 6 models (Llama-3.2-Vision, Pixtral, IDEFICS, Idefics2, InstructBLIP, Chameleon) will 403 on download.


---
## 6. Disk space

The full HF cache for 20 models is approximately 250-300 GB. Because HF stores blobs + symlinks, the on-disk footprint can be roughly 2× that. **Minimum 500 GB recommended; 1 TB+ for headroom.** Colab's free tier (235 GB) will run out around model 14 (InternVL3-8B).

In [7]:
free_gb, total_gb = (lambda s: (s.free/1e9, s.total/1e9))(shutil.disk_usage('/'))
used_gb = total_gb - free_gb
print(f'Disk:  {free_gb:.1f} GB free / {total_gb:.1f} GB total ({100*free_gb/total_gb:.1f}% free, {used_gb:.1f} GB used)')

if free_gb < 100:
    print(f'⛔ {free_gb:.0f} GB free — you will run out of disk before model 10. ABORT and resize the volume.')
elif free_gb < 300:
    print(f'⚠ {free_gb:.0f} GB free — enough for ~half the roster. Set CLEANUP_HF_CACHE_BETWEEN_MODELS=True in notebook 02.')
elif free_gb < 500:
    print(f'⚠ {free_gb:.0f} GB free — tight. Should fit all 20 with no headroom for FLOPs/memory caches.')
else:
    print(f'✓ {free_gb:.0f} GB free — comfortable for the full 20-model pipeline.')

# HF cache location and persistence check
HF_HOME = Path(os.environ.get('HF_HOME', str(Path.home() / '.cache' / 'huggingface'))).expanduser()
HF_HUB = HF_HOME / 'hub'
print(f'\nHF_HOME env:  {os.environ.get("HF_HOME", "<not set, using default>")}')
print(f'HF_HOME path: {HF_HOME}')
print(f'HF cache:     {HF_HUB}  ({"exists" if HF_HUB.exists() else "will be created on first download"})')

# Persistence: on Runpod, /root is ephemeral (wiped on pod restart);
# only /workspace is persistent — and only if you mounted a Network
# Volume there. If HF_HOME is under /root, you'll re-download all 250+
# GB every time the pod restarts.
_p = str(HF_HOME)
if _p.startswith('/root'):
    print('⛔ HF_HOME is under /root — EPHEMERAL on Runpod. Every pod restart wipes the cache.')
    print('   Fix (Runpod):')
    print('     1) Stop the pod')
    print('     2) Edit the pod, add an env var:  HF_HOME=/workspace/.cache/huggingface')
    print('     3) Make sure /workspace is backed by a Network Volume (not a Container Volume)')
    print('     4) Start the pod, re-run this notebook to confirm HF_HOME now reads /workspace/...')
elif _p.startswith('/workspace'):
    print('✓ HF_HOME is under /workspace — persistent IF backed by a Runpod Network Volume.')
    print('   Sanity check: run `df -h /workspace` in a terminal — size should match your Network Volume size,')
    print('   not the small Container Disk default (often 20 GB).')
else:
    print(f'? HF_HOME at {_p} — confirm this is on a persistent volume in your environment.')

if HF_HUB.exists():
    n_models = len([p for p in HF_HUB.iterdir() if p.is_dir() and p.name.startswith('models--')])
    cached_gb = 0.0
    for p in HF_HUB.rglob('*'):
        if p.is_file() and not p.is_symlink():
            try:
                cached_gb += p.stat().st_size / 1e9
            except Exception:
                pass
    print(f'Models already cached: {n_models}  ({cached_gb:.1f} GB on disk)')

Disk:  49.9 GB free / 53.7 GB total (92.9% free, 3.8 GB used)
⛔ 50 GB free — you will run out of disk before model 10. ABORT and resize the volume.

HF_HOME env:  /workspace/.cache/huggingface/
HF_HOME path: /workspace/.cache/huggingface
HF cache:     /workspace/.cache/huggingface/hub  (exists)
✓ HF_HOME is under /workspace — persistent IF backed by a Runpod Network Volume.
   Sanity check: run `df -h /workspace` in a terminal — size should match your Network Volume size,
   not the small Container Disk default (often 20 GB).
Models already cached: 1  (0.5 GB on disk)


---
## 7. Output directories

In [8]:
MEASUREMENT_DIR = CODE_ACTIVE / 'outputs' / 'measurements'
LOG_DIR = CODE_ACTIVE / 'outputs' / 'logs'
FIG_DIR = CODE_ACTIVE / 'outputs' / 'figures'
for d in (MEASUREMENT_DIR, LOG_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)
    print(f'✓ {d}  ({"empty" if not any(d.iterdir()) else str(len(list(d.iterdir())))+" existing files"})')

✓ /workspace/code_active/outputs/measurements  (1 existing files)
✓ /workspace/code_active/outputs/logs  (2 existing files)
✓ /workspace/code_active/outputs/figures  (empty)


---
## 8. vlm_audit package importability

In [9]:
from vlm_audit import (
    MODELS, get_model_list, count_params,
    extract_components, compute_trainable_from_policy,
    run_full_audit, InputConfig, ModelRecord, ComponentRecord,
)
from vlm_audit.components import (
    COMPONENT_NAMES, EXTENDED_COMPONENT_NAMES,
    freezing_policy_hint, trainable_components_for_family,
)
from vlm_audit.metrics import (
    measure_flops, measure_peak_memory,
    count_fusion_boundary_tokens, count_active_params,
)
from vlm_audit.runner import _resolve_hf_class, _load_model
from vlm_audit.dummy_inputs import build_dummy_inputs
from vlm_audit.logging_utils import ExperimentLogger, sanitize_for_filename
from vlm_audit.quantization import get_quantization_config

print(f'✓ vlm_audit imports OK')
print(f'  MODELS:            {len(MODELS)} model specs')
print(f'  COMPONENT_NAMES:   {COMPONENT_NAMES}')
print(f'  EXTENDED_NAMES:    {EXTENDED_COMPONENT_NAMES}')
print(f'  Default InputConfig: {InputConfig().to_dict()}')

✓ vlm_audit imports OK
  MODELS:            20 model specs
  COMPONENT_NAMES:   ('vision', 'fusion', 'language', 'head')
  EXTENDED_NAMES:    ('vision', 'fusion', 'language', 'head', 'fused_transformer', 'vision_und', 'vision_gen', 'embeddings')
  Default InputConfig: {'image_hw': [224, 224], 'text_tokens': 32, 'batch_size': 1, 'precision': 'fp16'}


---
## 9. Roster sanity

Quick view of all 20 models. If the count is wrong, your `models.py` is out of date.

In [10]:
import pandas as pd
roster = pd.DataFrame([
    dict(name=m.name, family=m.family, year=m.release_year,
         fp16_GB=m.fits_in_gb_fp16, native_HW=f'{m.native_image_hw[0]}x{m.native_image_hw[1]}',
         recent_gen=m.recent_gen, is_moe=m.is_moe, trust_remote=m.requires_trust_remote_code,
         hf_id=m.hf_id)
    for m in MODELS
])
assert len(roster) == 20, f'Expected 20 models, got {len(roster)} — check vlm_audit/models.py'
print(f'✓ 20 models in roster, {len(roster.family.unique())} unique families')
print(f'Cumulative fp16 footprint: {roster.fp16_GB.sum():.0f} GB')
roster

✓ 20 models in roster, 13 unique families
Cumulative fp16 footprint: 266 GB


,name,family,year,fp16_GB,native_HW,recent_gen,is_moe,trust_remote,hf_id
0,CLIP-ViT-L/14,dual_encoder_contrastive,2021,1.0,224x224,False,False,False,openai/clip-vit-large-patch14
1,SigLIP-L/16,dual_encoder_contrastive,2023,1.5,384x384,False,False,False,google/siglip-large-patch16-384
2,BLIP-large,bridge_module_precursor,2022,1.0,384x384,False,False,False,Salesforce/blip-image-captioning-large
3,BLIP-2-OPT-2.7B,bridge_module,2023,7.0,224x224,False,False,False,Salesforce/blip2-opt-2.7b
4,InstructBLIP-Vicuna-7B,bridge_module_adapter_llm,2023,17.0,224x224,False,False,False,Salesforce/instructblip-vicuna-7b
5,LLaVA-1.5-7B,adapter_llm,2023,16.0,336x336,False,False,False,llava-hf/llava-1.5-7b-hf
6,LLaVA-NeXT-7B,adapter_llm,2024,16.0,336x336,True,False,False,llava-hf/llava-v1.6-mistral-7b-hf
7,IDEFICS-9B,cross_attention_llm,2023,18.0,224x224,False,False,False,HuggingFaceM4/idefics-9b
8,Idefics2-8B,perceiver_adapter_llm,2024,16.0,224x224,True,False,False,HuggingFaceM4/idefics2-8b
9,ViLT-B/32,single_stream_fused,2021,0.5,384x384,False,False,False,dandelin/vilt-b32-finetuned-vqa


---
## 10. Mini smoke test — ViLT-B/32 (smallest model, 117M params, 460 MB download)

End-to-end sanity check: load the smallest model, extract components, count params. If this works, the pipeline is healthy for everything else.

*Why ViLT?* It's the smallest model in the roster (117M params, ~460 MB download) so the smoke test completes in under a minute. ViLT also exercises the `single_stream_fused` family extractor, which is one of the simpler ones.

In [11]:
import time
vilt_spec = next(m for m in MODELS if m.name == 'ViLT-B/32')
print(f'Loading {vilt_spec.name} ({vilt_spec.hf_id})...')
t0 = time.time()
try:
    rec = run_full_audit(
        vilt_spec, metrics={'params'},
        measurement_dir=str(MEASUREMENT_DIR),
        log_dir=str(LOG_DIR),
        notebook_label='00_setup_check_nb',
    )
    elapsed = time.time() - t0
    print(f'✓ Smoke test passed in {elapsed:.1f}s')
    for cname, c in rec.components.items():
        total = c.params_total or 0
        if total > 0:
            print(f'  {cname:10s}: {total:>12,}  ({total/1e6:.2f}M)')
    total_all = sum((c.params_total or 0) for c in rec.components.values())
    print(f'  {"total":10s}: {total_all:>12,}  ({total_all/1e6:.2f}M)')
    # ViLT-B/32-finetuned-VQA expected total ≈ 117M params
    assert 100_000_000 < total_all < 140_000_000, f'ViLT total {total_all:,} outside expected 100-140M range'
    print('  ✓ Total matches expected ViLT-B/32 size (~117M)')
except Exception as e:
    import traceback
    print(f'⛔ Smoke test FAILED: {type(e).__name__}: {e}')
    traceback.print_exc()

Loading ViLT-B/32 (dandelin/vilt-b32-finetuned-vqa)...


model.safetensors:   0%|          | 0.00/470M [00:00<?, ?B/s]

✓ Smoke test passed in 3.8s
  vision    :   25,948,416  (25.95M)
  language  :   85,054,464  (85.05M)
  head      :    5,993,529  (5.99M)
  total     :  116,996,409  (117.00M)
  ✓ Total matches expected ViLT-B/32 size (~117M)


---
## 11. Pre-flight summary

Re-print everything in one block so it's easy to copy-paste for a Runpod log or a status update.

In [13]:
print('=' * 70)
print('PRE-FLIGHT SUMMARY')
print('=' * 70)
print(f'Working dir:      {NB_DIR}')
print(f'PyTorch:          {torch.__version__}  (CUDA={torch.cuda.is_available()})')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU:              {p.name}  ({p.total_memory/1e9:.1f} GB VRAM)')
print(f'transformers:     {transformers.__version__}')
print(f'FLOPs counters:   {flops_chain or "NONE"}')
print(f'HF_TOKEN set:     {bool(hf_token)}')
print(f'Disk free:        {free_gb:.0f} GB / {total_gb:.0f} GB')
print(f'Models in roster: {len(MODELS)}')
print(f'Smoke test:       {"PASSED" if "rec" in dir() and rec else "FAILED"}')
print('=' * 70)
print('Next: open 02_parameter_analysis_v3.ipynb and run the full pipeline.')

PRE-FLIGHT SUMMARY
Working dir:      /workspace/code_active/notebooks
PyTorch:          2.8.0+cu128  (CUDA=True)
GPU:              NVIDIA A100-SXM4-80GB  (85.1 GB VRAM)
transformers:     4.57.6
FLOPs counters:   ['calflops', 'fvcore', 'ptflops']
HF_TOKEN set:     False
Disk free:        50 GB / 54 GB
Models in roster: 20
Smoke test:       PASSED
Next: open 02_parameter_analysis_v3.ipynb and run the full pipeline.


In [17]:
!nvidia-smi

Fri May 22 08:32:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.195.03             Driver Version: 570.195.03     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          On  |   00000000:DB:00.0 Off |                    0 |
| N/A   31C    P0             85W /  400W |    1013MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----